# Comprehensive Google ClusterData2019 + PowerData2019 Extraction

**Purpose:** Extract all datasets needed to benchmark a convex-optimization-based 
green data center scheduler against three papers from the Lin et al. (2024) survey:

| Paper | Key Result | What We Extract |
|-------|-----------|-----------------|
| **Grange et al. (2018)** — Batch scheduling w/ renewable awareness | 49% brown-energy ↓, 51% cost ↓ | Batch job statistical profiles (distributions for workload generator) |
| **Xu et al. (2020)** — Self-adaptive brownout + batch deferral | 21% brown-energy ↓, 10% renewable ↑ | Mixed batch/service classification, aggregate utilization curves |
| **Haghshenas et al. (2022)** — Infrastructure-aware heterogeneous scheduling | Energy-cost minimization | Heterogeneous machine fleet, cooling-relevant utilization, workload mix |

## Datasets Produced

| # | Dataset | File(s) | Size | Used By |
|---|---------|---------|------|---------|
| 1 | Cell CPU utilization (5-min) | `cells/cell_{a..d}.csv` | ~50K rows/cell | All three |
| 2 | Power model (CPU→Power) | `power_model_params.json`, `power_model_scatter.csv` | <1 MB | All three |
| 3 | Machine attributes (heterogeneous fleet) | `machines/machines_{a..d}.csv`, `machines_all.csv` | ~100K rows total | Haghshenas |
| 4 | Job metadata + batch/service classification | `jobs/jobs_{a..d}.csv` | ~1M rows/cell | Xu, Haghshenas |
| 5 | **Batch job statistical profiles** | `jobs/batch_distributions_{a..d}.json` | <100 KB | Grange |
| 6 | **Workload generator parameters** | `workload_generator_params.json` | <10 KB | Grange |

## Methodology Note

Following Grange et al.'s approach (which uses the Da Costa et al. 2016 workload generator), 
we do **not** replay raw instance events. Instead we:
1. Fit statistical distributions to real job characteristics (inter-arrival times, durations, resource requests)
2. Export the fitted parameters for use in a synthetic workload generator
3. Validate the fits with KS-test statistics

This is standard practice in the scheduling literature and produces a stronger experimental 
design than raw replay because it allows controlled variation of load intensity and job mix.

**Before running:** GCP project with BigQuery API enabled → https://console.cloud.google.com/flows/enableapi?apiid=bigquery

In [ ]:
# ============================================================
# Step 1: Authenticate and configure
# ============================================================
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'YOUR_PROJECT_ID_HERE'  # <-- CHANGE THIS

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Quick test
test_q = """
SELECT SUM(cpu_cap) AS cpu_capacity
FROM (
    SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
    FROM `google.com:google-cluster-data`.clusterdata_2019_a.machine_events
    GROUP BY 1
)
"""
result = client.query(test_q).to_dataframe()
print(f"Cell 'a' CPU capacity: {result['cpu_capacity'].iloc[0]:.2f}")
print(f'Authenticated with project: {PROJECT_ID}  ✓')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from scipy import stats

os.makedirs('data/cells', exist_ok=True)
os.makedirs('data/jobs', exist_ok=True)
os.makedirs('data/machines', exist_ok=True)

CELLS = ['a', 'b', 'c', 'd']

def get_cell_capacity(cell):
    """Get total CPU capacity for a cell."""
    query = f"""
    SELECT SUM(cpu_cap) AS cpu_capacity
    FROM (
        SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1
    )
    """
    return float(client.query(query).to_dataframe()['cpu_capacity'].iloc[0])

# Pre-fetch capacities (reused by multiple datasets)
cell_capacities = {}
for cell in CELLS:
    cell_capacities[cell] = get_cell_capacity(cell)
    print(f"Cell {cell} CPU capacity: {cell_capacities[cell]:.2f}")

---
## Dataset 1: Cell-Level CPU Utilization (5-min intervals)

Aggregate normalized CPU demand per cell at 5-minute resolution.  
Used as the **demand curve input** for all three benchmark papers' optimization models.

In [ ]:
for cell in CELLS:
    print(f'\n--- Cell {cell}: CPU Utilization ---')
    cap = cell_capacities[cell]

    query = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 300)) AS INT64) AS time_bucket,
        SUM(average_usage.cpus) / {cap} AS cpu_demand_norm
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['timestep'] = df['time_bucket'] - df['time_bucket'].min()
    df = df[['timestep', 'cpu_demand_norm']].copy()
    df['cpu_demand_norm'] = df['cpu_demand_norm'].clip(0.0, 1.0)

    path = f'data/cells/cell_{cell}.csv'
    df.to_csv(path, index=False)
    print(f'  {len(df)} rows → {path}')
    print(f'  Mean={df["cpu_demand_norm"].mean():.4f}, Max={df["cpu_demand_norm"].max():.4f}')

print('\n✅ Cell utilization done!')

---
## Dataset 2: Power Model (CPU → Power)

Fits a linear model `P = P_idle + slope × cpu_util` from Google PowerData2019.  
This is the standard server power model used across the sustainable DC literature.

In [ ]:
print('Extracting hourly power utilization...')
power_query = """
SELECT
    cell,
    CAST(FLOOR(time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
    AVG(measured_power_util) AS avg_power_util
FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
WHERE NOT bad_measurement_data
    AND cell IN ('a', 'b', 'c', 'd')
GROUP BY 1, 2
ORDER BY 1, 2
"""
power_df = client.query(power_query).to_dataframe()
print(f'  {len(power_df)} power rows')

all_cpu = []
for cell in CELLS:
    print(f'  Hourly CPU for cell {cell}...')
    cap = cell_capacities[cell]
    q = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
        SUM(average_usage.cpus) / (12 * {cap}) AS avg_cpu_util
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    cpu_df = client.query(q).to_dataframe()
    cpu_df['cell'] = cell
    all_cpu.append(cpu_df)

cpu_combined = pd.concat(all_cpu, ignore_index=True)
merged = pd.merge(cpu_combined, power_df, on=['cell', 'hour_index'], how='inner')

cpu_util = merged['avg_cpu_util'].values
power_util = merged['avg_power_util'].values
valid = np.isfinite(cpu_util) & np.isfinite(power_util)
cpu_util, power_util = cpu_util[valid], power_util[valid]

A = np.vstack([np.ones_like(cpu_util), cpu_util]).T
result = np.linalg.lstsq(A, power_util, rcond=None)
intercept, slope = result[0]
predicted = intercept + slope * cpu_util
ss_res = np.sum((power_util - predicted) ** 2)
ss_tot = np.sum((power_util - power_util.mean()) ** 2)
r_squared = 1.0 - ss_res / ss_tot

params = {
    'idle_power': float(intercept),
    'peak_power': float(intercept + slope),
    'slope': float(slope),
    'r_squared': float(r_squared),
    'description': 'Linear power model: P = idle_power + slope * cpu_utilization'
}
print(f'\nPower Model: P_idle={params["idle_power"]:.4f}  P_peak={params["peak_power"]:.4f}  '
      f'slope={params["slope"]:.4f}  R²={params["r_squared"]:.4f}')

with open('data/power_model_params.json', 'w') as f:
    json.dump(params, f, indent=2)
pd.DataFrame({'cpu_util': cpu_util, 'power_util': power_util}).to_csv(
    'data/power_model_scatter.csv', index=False)

print('✅ Power model done!')

---
## Dataset 3: Machine Attributes (Heterogeneous Fleet)

Per-machine CPU and memory capacity for modeling heterogeneous server infrastructure.  
**Haghshenas et al.** use this to model infrastructure-aware scheduling where different
server types have different power profiles and capabilities.

In [ ]:
all_machines = []
for cell in CELLS:
    print(f'\n--- Cell {cell}: Machines ---')
    query = f"""
    SELECT
        machine_id,
        MAX(capacity.cpus) AS cpu_capacity,
        MAX(capacity.memory) AS memory_capacity
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
    GROUP BY 1
    HAVING cpu_capacity IS NOT NULL AND memory_capacity IS NOT NULL
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['cell'] = cell
    df.to_csv(f'data/machines/machines_{cell}.csv', index=False)
    all_machines.append(df)

    print(f'  {len(df)} machines')
    print(f'  CPU  — unique types: {df["cpu_capacity"].nunique()}, '
          f'range: [{df["cpu_capacity"].min():.4f}, {df["cpu_capacity"].max():.4f}]')
    print(f'  Mem  — unique types: {df["memory_capacity"].nunique()}, '
          f'range: [{df["memory_capacity"].min():.4f}, {df["memory_capacity"].max():.4f}]')

combined = pd.concat(all_machines, ignore_index=True)
combined.to_csv('data/machines/machines_all.csv', index=False)
print(f'\n✅ {len(combined)} total machines across all cells')

---
## Dataset 4: Job Metadata + Batch/Service Classification

Job-level metadata from `collection_events` joined with aggregate resource requests
from `instance_events`. Each job is classified as **batch** or **service** using 
Google Borg conventions:

- `scheduling_class ≤ 1 AND priority < 200` → **Batch** (delay-tolerant)
- Otherwise → **Service** (latency-sensitive)

**Xu et al.** use this split for their brownout (service) + deferral (batch) algorithms.  
**Haghshenas et al.** use the heterogeneous workload mix for infrastructure-aware scheduling.

In [ ]:
for cell in CELLS:
    print(f'\n--- Cell {cell}: Job Metadata ---')

    query = f"""
    WITH submits AS (
        SELECT
            collection_id,
            MIN(time) AS submit_time,
            MAX(scheduling_class) AS scheduling_class,
            MAX(priority) AS priority,
            CASE
                WHEN MAX(scheduling_class) <= 1 AND MAX(priority) < 200 THEN 'batch'
                ELSE 'service'
            END AS job_type
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.collection_events
        WHERE type = 0
        GROUP BY collection_id
    ),
    terminals AS (
        SELECT
            collection_id,
            MIN(time) AS end_time,
            MIN(type) AS terminal_type
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.collection_events
        WHERE type IN (4, 5, 6)
        GROUP BY collection_id
    ),
    resources AS (
        SELECT
            collection_id,
            COUNT(DISTINCT instance_index) AS num_tasks,
            AVG(resource_request.cpus) AS avg_cpu_request,
            AVG(resource_request.memory) AS avg_mem_request,
            SUM(resource_request.cpus) AS total_cpu_request,
            SUM(resource_request.memory) AS total_mem_request
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_events
        WHERE type = 0
        GROUP BY collection_id
    )
    SELECT
        s.collection_id,
        s.submit_time,
        t.end_time,
        (t.end_time - s.submit_time) / 1e6 AS duration_sec,
        s.scheduling_class,
        s.priority,
        s.job_type,
        t.terminal_type,
        r.num_tasks,
        r.avg_cpu_request,
        r.avg_mem_request,
        r.total_cpu_request,
        r.total_mem_request
    FROM submits s
    LEFT JOIN terminals t ON s.collection_id = t.collection_id
    LEFT JOIN resources r ON s.collection_id = r.collection_id
    WHERE s.submit_time IS NOT NULL
    ORDER BY s.submit_time
    """
    df = client.query(query).to_dataframe()
    df.to_csv(f'data/jobs/jobs_{cell}.csv', index=False)

    n_batch = (df['job_type'] == 'batch').sum()
    n_service = (df['job_type'] == 'service').sum()
    print(f'  {len(df)} jobs — {n_batch} batch ({100*n_batch/len(df):.1f}%), '
          f'{n_service} service ({100*n_service/len(df):.1f}%)')

    completed = df[df['duration_sec'].notna() & (df['duration_sec'] > 0)]
    if len(completed) > 0:
        print(f'  Completed: {len(completed)} — '
              f'duration median={completed["duration_sec"].median():.0f}s, '
              f'mean={completed["duration_sec"].mean():.0f}s')

print('\n✅ Job metadata done!')

---
## Dataset 5: Batch Job Statistical Profiles (Distribution Fitting)

This is the key methodological improvement over raw event replay.  
Following **Grange et al.** and **Da Costa et al. (2016)**, we fit statistical 
distributions to three batch job characteristics:

1. **Inter-arrival times** — time between consecutive batch job submissions
2. **Job durations** — wall-clock time from submit to completion
3. **Resource requests** — CPU and memory per task

For each, we fit candidate distributions (exponential, lognormal, gamma, Weibull)
and select the best fit by KS-test. The fitted parameters are exported for use 
in a synthetic workload generator.

### Why this approach?
- Allows controlled experiments: vary load intensity, job mix, flexibility factor
- Standard methodology in scheduling literature (reproducible)
- Avoids trace artifacts (cold-start, Borg-internal retries, alloc-set nesting)
- Produces a cleaner thesis methodology section

In [ ]:
def fit_best_distribution(data, name, candidates=None):
    """Fit candidate distributions, return best by KS-test p-value."""
    if candidates is None:
        candidates = ['expon', 'lognorm', 'gamma', 'weibull_min']

    data = data[np.isfinite(data) & (data > 0)]
    if len(data) < 100:
        return {'distribution': 'insufficient_data', 'n_samples': len(data)}

    best = {'ks_pvalue': -1}
    for dist_name in candidates:
        try:
            dist = getattr(stats, dist_name)
            params = dist.fit(data)
            ks_stat, ks_pval = stats.kstest(data, dist_name, args=params)
            if ks_pval > best['ks_pvalue']:
                best = {
                    'distribution': dist_name,
                    'params': [float(p) for p in params],
                    'ks_statistic': float(ks_stat),
                    'ks_pvalue': float(ks_pval),
                }
        except Exception:
            continue

    # Also store summary statistics for sanity checks
    best.update({
        'name': name,
        'n_samples': len(data),
        'mean': float(np.mean(data)),
        'median': float(np.median(data)),
        'std': float(np.std(data)),
        'p5': float(np.percentile(data, 5)),
        'p25': float(np.percentile(data, 25)),
        'p75': float(np.percentile(data, 75)),
        'p95': float(np.percentile(data, 95)),
    })
    return best


all_cell_profiles = {}

for cell in CELLS:
    print(f'\n{"="*50}')
    print(f'Cell {cell}: Fitting batch job distributions')
    print(f'{"="*50}')

    # Load the job metadata we already extracted
    jobs = pd.read_csv(f'data/jobs/jobs_{cell}.csv')
    batch = jobs[(jobs['job_type'] == 'batch')].copy()
    completed_batch = batch[batch['duration_sec'].notna() & (batch['duration_sec'] > 0)].copy()

    print(f'  Total batch jobs: {len(batch)}')
    print(f'  Completed batch jobs: {len(completed_batch)}')

    if len(completed_batch) < 100:
        print(f'  ⚠ Too few completed batch jobs, skipping distribution fitting')
        continue

    profiles = {}

    # 1. Inter-arrival times
    submit_times = np.sort(completed_batch['submit_time'].values)
    inter_arrivals = np.diff(submit_times) / 1e6  # convert μs → seconds
    inter_arrivals = inter_arrivals[inter_arrivals > 0]
    fit = fit_best_distribution(inter_arrivals, 'inter_arrival_sec')
    profiles['inter_arrival'] = fit
    print(f'\n  Inter-arrival times:')
    print(f'    Best fit: {fit["distribution"]} (KS p={fit.get("ks_pvalue",0):.4f})')
    print(f'    Mean={fit["mean"]:.1f}s, Median={fit["median"]:.1f}s')

    # 2. Job durations
    durations = completed_batch['duration_sec'].values
    fit = fit_best_distribution(durations, 'duration_sec')
    profiles['duration'] = fit
    print(f'\n  Job durations:')
    print(f'    Best fit: {fit["distribution"]} (KS p={fit.get("ks_pvalue",0):.4f})')
    print(f'    Mean={fit["mean"]:.0f}s, Median={fit["median"]:.0f}s')

    # 3. CPU request per task
    cpu_req = completed_batch['avg_cpu_request'].dropna().values
    fit = fit_best_distribution(cpu_req, 'cpu_request')
    profiles['cpu_request'] = fit
    print(f'\n  CPU request per task:')
    print(f'    Best fit: {fit["distribution"]} (KS p={fit.get("ks_pvalue",0):.4f})')
    print(f'    Mean={fit["mean"]:.4f}, Median={fit["median"]:.4f}')

    # 4. Memory request per task
    mem_req = completed_batch['avg_mem_request'].dropna().values
    fit = fit_best_distribution(mem_req, 'memory_request')
    profiles['memory_request'] = fit
    print(f'\n  Memory request per task:')
    print(f'    Best fit: {fit["distribution"]} (KS p={fit.get("ks_pvalue",0):.4f})')
    print(f'    Mean={fit["mean"]:.4f}, Median={fit["median"]:.4f}')

    # 5. Tasks per job
    tasks_per_job = completed_batch['num_tasks'].dropna().values
    fit = fit_best_distribution(tasks_per_job.astype(float), 'tasks_per_job')
    profiles['tasks_per_job'] = fit
    print(f'\n  Tasks per job:')
    print(f'    Best fit: {fit["distribution"]} (KS p={fit.get("ks_pvalue",0):.4f})')
    print(f'    Mean={fit["mean"]:.1f}, Median={fit["median"]:.1f}')

    # 6. Workload mix ratio
    total_jobs = len(jobs)
    profiles['workload_mix'] = {
        'total_jobs': int(total_jobs),
        'batch_count': int((jobs['job_type'] == 'batch').sum()),
        'service_count': int((jobs['job_type'] == 'service').sum()),
        'batch_fraction': float((jobs['job_type'] == 'batch').mean()),
    }
    print(f'\n  Workload mix: {profiles["workload_mix"]["batch_fraction"]:.1%} batch')

    # Save per-cell profile
    with open(f'data/jobs/batch_distributions_{cell}.json', 'w') as f:
        json.dump(profiles, f, indent=2)
    all_cell_profiles[cell] = profiles

print('\n✅ Distribution fitting done!')

---
## Dataset 6: Unified Workload Generator Parameters

Aggregates the per-cell distribution fits into a single parameter file
that can drive a synthetic workload generator. Also includes the 
flexibility-factor methodology from Grange et al. for generating deadlines:

```
deadline = submit_time + duration × (1 + flexibility_factor)
```

Where `flexibility_factor ∈ [0.5, 1.0, 2.0, 4.0]` controls how much slack
batch jobs have — this is a key experimental variable in Grange et al.

In [ ]:
# Aggregate across cells — use cell 'a' as primary (largest, most representative)
# but include all cells for cross-validation
primary_cell = 'a'

if primary_cell in all_cell_profiles:
    p = all_cell_profiles[primary_cell]

    generator_params = {
        'description': (
            'Workload generator parameters fitted to Google ClusterData2019. '
            'Distributions fitted via MLE with KS-test selection. '
            'Following Da Costa et al. (2016) and Grange et al. (2018) methodology.'
        ),
        'primary_cell': primary_cell,
        'inter_arrival': p['inter_arrival'],
        'duration': p['duration'],
        'cpu_request': p['cpu_request'],
        'memory_request': p['memory_request'],
        'tasks_per_job': p['tasks_per_job'],
        'workload_mix': p['workload_mix'],
        'deadline_model': {
            'description': 'deadline = submit_time + duration * (1 + flexibility_factor)',
            'flexibility_factors': [0.5, 1.0, 2.0, 4.0],
            'note': 'Grange et al. sweep these values; higher = more scheduling slack'
        },
        'cross_validation': {
            cell: {
                'batch_fraction': all_cell_profiles[cell]['workload_mix']['batch_fraction'],
                'mean_duration': all_cell_profiles[cell]['duration']['mean'],
                'mean_inter_arrival': all_cell_profiles[cell]['inter_arrival']['mean'],
            }
            for cell in all_cell_profiles
        }
    }

    with open('data/workload_generator_params.json', 'w') as f:
        json.dump(generator_params, f, indent=2)
    print('Saved data/workload_generator_params.json')

    print(f'\nPrimary cell ({primary_cell}) generator parameters:')
    print(f'  Inter-arrival: {p["inter_arrival"]["distribution"]} '
          f'(mean={p["inter_arrival"]["mean"]:.1f}s)')
    print(f'  Duration:      {p["duration"]["distribution"]} '
          f'(mean={p["duration"]["mean"]:.0f}s)')
    print(f'  CPU request:   {p["cpu_request"]["distribution"]} '
          f'(mean={p["cpu_request"]["mean"]:.4f})')
    print(f'  Tasks/job:     {p["tasks_per_job"]["distribution"]} '
          f'(mean={p["tasks_per_job"]["mean"]:.1f})')
    print(f'  Batch fraction: {p["workload_mix"]["batch_fraction"]:.1%}')

    print(f'\nCross-cell validation:')
    for cell in all_cell_profiles:
        cv = generator_params['cross_validation'][cell]
        print(f'  Cell {cell}: batch={cv["batch_fraction"]:.1%}, '
              f'dur_mean={cv["mean_duration"]:.0f}s, '
              f'iat_mean={cv["mean_inter_arrival"]:.1f}s')
else:
    print('⚠ Primary cell profiles not available — check Dataset 4 and 5 outputs')

print('\n✅ Workload generator params done!')

---
## Validation Plots

Visual sanity checks: distribution fits, utilization curves, machine heterogeneity.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('ClusterData2019 Extraction Summary', fontsize=14, fontweight='bold')

# 1. Cell utilization (cell a)
ax = axes[0, 0]
cell_df = pd.read_csv('data/cells/cell_a.csv')
# Downsample for plotting
step = max(1, len(cell_df) // 2000)
ax.plot(cell_df['timestep'].values[::step] / 12 / 24,
        cell_df['cpu_demand_norm'].values[::step], linewidth=0.5, alpha=0.8)
ax.set_xlabel('Days')
ax.set_ylabel('CPU Utilization')
ax.set_title('Cell A: CPU Demand (5-min)')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# 2. Power model
ax = axes[0, 1]
scatter = pd.read_csv('data/power_model_scatter.csv')
ax.scatter(scatter['cpu_util'], scatter['power_util'], alpha=0.05, s=1)
with open('data/power_model_params.json') as f:
    pm = json.load(f)
x_line = np.linspace(0, scatter['cpu_util'].max(), 100)
ax.plot(x_line, pm['idle_power'] + pm['slope'] * x_line, 'r-', linewidth=2,
        label=f'P={pm["idle_power"]:.3f}+{pm["slope"]:.3f}×CPU\nR²={pm["r_squared"]:.3f}')
ax.set_xlabel('CPU Utilization')
ax.set_ylabel('Power Utilization')
ax.set_title('Power Model Fit')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Machine heterogeneity
ax = axes[0, 2]
machines = pd.read_csv('data/machines/machines_all.csv')
ax.scatter(machines['cpu_capacity'], machines['memory_capacity'],
           alpha=0.1, s=2, c=machines['cell'].map({'a':0,'b':1,'c':2,'d':3}), cmap='tab10')
ax.set_xlabel('CPU Capacity (normalized)')
ax.set_ylabel('Memory Capacity (normalized)')
ax.set_title(f'Machine Fleet ({len(machines)} servers)')
ax.grid(True, alpha=0.3)

# 4. Job duration distribution (batch, cell a)
ax = axes[1, 0]
jobs_a = pd.read_csv('data/jobs/jobs_a.csv')
batch_dur = jobs_a[(jobs_a['job_type']=='batch') &
                   (jobs_a['duration_sec'] > 0) &
                   (jobs_a['duration_sec'].notna())]['duration_sec']
if len(batch_dur) > 0:
    ax.hist(np.log10(batch_dur.clip(lower=1)), bins=80, density=True, alpha=0.7, color='steelblue')
    ax.set_xlabel('log₁₀(Duration / seconds)')
    ax.set_ylabel('Density')
ax.set_title(f'Batch Job Durations (Cell A, n={len(batch_dur)})')
ax.grid(True, alpha=0.3)

# 5. Workload mix across cells
ax = axes[1, 1]
mix_data = []
for cell in CELLS:
    jf = pd.read_csv(f'data/jobs/jobs_{cell}.csv')
    mix_data.append({
        'cell': cell,
        'batch': (jf['job_type']=='batch').sum(),
        'service': (jf['job_type']=='service').sum()
    })
mix_df = pd.DataFrame(mix_data)
x = range(len(mix_df))
ax.bar(x, mix_df['batch'], label='Batch', color='steelblue')
ax.bar(x, mix_df['service'], bottom=mix_df['batch'], label='Service', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([f'Cell {c}' for c in CELLS])
ax.set_ylabel('Number of Jobs')
ax.set_title('Workload Mix by Cell')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 6. Inter-arrival time distribution (cell a)
ax = axes[1, 2]
batch_a = jobs_a[(jobs_a['job_type']=='batch') & jobs_a['submit_time'].notna()].sort_values('submit_time')
iat = np.diff(batch_a['submit_time'].values) / 1e6  # μs → seconds
iat = iat[(iat > 0) & (iat < np.percentile(iat[iat>0], 99))]
if len(iat) > 0:
    ax.hist(iat, bins=100, density=True, alpha=0.7, color='steelblue')
    ax.set_xlabel('Inter-arrival Time (seconds)')
    ax.set_ylabel('Density')
ax.set_title(f'Batch Inter-arrival (Cell A, n={len(iat)})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/extraction_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved data/extraction_summary.png')

---
## Summary & Download

In [ ]:
print('=' * 65)
print('EXTRACTION SUMMARY')
print('=' * 65)

for cell in CELLS:
    print(f'\nCell {cell}:')
    for label, path in [
        ('Utilization (5min)', f'data/cells/cell_{cell}.csv'),
        ('Machines',           f'data/machines/machines_{cell}.csv'),
        ('Jobs (all)',         f'data/jobs/jobs_{cell}.csv'),
        ('Batch distributions', f'data/jobs/batch_distributions_{cell}.json'),
    ]:
        if os.path.exists(path):
            if path.endswith('.csv'):
                n = len(pd.read_csv(path))
                print(f'  {label:25s} {n:>10,} rows')
            else:
                size = os.path.getsize(path)
                print(f'  {label:25s} {size/1024:>10.1f} KB')

print(f'\nGlobal files:')
for label, path in [
    ('Power model',           'data/power_model_params.json'),
    ('All machines',          'data/machines/machines_all.csv'),
    ('Generator params',      'data/workload_generator_params.json'),
    ('Power scatter',         'data/power_model_scatter.csv'),
    ('Summary plot',          'data/extraction_summary.png'),
]:
    if os.path.exists(path):
        if path.endswith('.csv'):
            n = len(pd.read_csv(path))
            print(f'  {label:25s} {n:>10,} rows')
        else:
            size = os.path.getsize(path)
            print(f'  {label:25s} {size/1024:>10.1f} KB')

print('\n' + '=' * 65)
print('BENCHMARK PAPER MAPPING')
print('=' * 65)
print("""
Grange et al. (2018) — Batch scheduling + renewable awareness:
  → workload_generator_params.json  (generate synthetic batch jobs)
  → cells/cell_*.csv               (aggregate demand for capacity)
  → power_model_params.json        (energy cost model)
  → YOU ADD: solar trace from NREL/PVGIS + electricity prices from ComEd/IESO
  → Deadline formula: submit_time + duration × (1 + flexibility_factor)

Xu et al. (2020) — Self-adaptive brownout + batch deferral:
  → jobs/jobs_*.csv                (batch/service classification + resource requests)
  → cells/cell_*.csv               (utilization time series for brownout triggers)
  → power_model_params.json        (energy model)
  → YOU ADD: renewable energy trace + brown/green energy pricing

Haghshenas et al. (2022) — Infrastructure-aware heterogeneous scheduling:
  → machines/machines_all.csv      (heterogeneous server fleet)
  → jobs/jobs_*.csv                (heterogeneous workload mix)
  → cells/cell_*.csv               (cooling model input — utilization drives heat)
  → power_model_params.json        (per-server-type power model)
  → YOU ADD: cooling power model + tiered electricity rate structure
""")

In [ ]:
# Download everything as zip
import shutil
from google.colab import files

shutil.make_archive('clusterdata2019_full', 'zip', '.', 'data')
files.download('clusterdata2019_full.zip')
print('Download started!')
print('Extract into C:\\Projects\\thesis\\ on your local machine.')